In [1]:
import os, textstat
from dotenv import load_dotenv
from google.genai import Client, types
from pydantic import BaseModel, Field
from typing import List, Dict

In [3]:
# retrieve API key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if api_key is None:
    raise ValueError("Gemini API key not set")

client = Client()

In [15]:
# output -> pydantic model for structured output -> Flesch-Kincaid score and SMOG score (before and after rewriting), simplified text
class ReadabilityEvaluation(BaseModel):
    pre_flesch_kincaid_grade: float = Field(description="Calculated Flesch-Kincaid grade level score before revision")
    post_flesch_kincaid_grade: float = Field(description="Calculated Flesch-Kincaid grade level score after revision")
    pre_smog: float = Field(description="SMOG readability score before revision")
    post_smog: float = Field(description="SMOG readability score after revision")
    revised_text: str = Field(description="Text rewritten at or below target grade level")

In [16]:
# agent prompt
# inputs -> text from Simplifier agent (confirm format), target grade level (8th)
# output -> defined pydantic model

def create_readability_chat(target_grade: int = 8):
    """Initializes chat session."""
    system_instruction = f"""
    You are an expert at evaluating the readability of clinical text and rewriting the text to be more readable.
    Evaluate the input text's readability using the Flesch-Kincaid grade level and SMOG metrics.
    Then, if the reading level required for the text is above {target_grade}th-grade level, rewrite the text
    to make it readable at the {target_grade}th-grade or below while maintaining the text's original meaning.

    Follow these guidelines:
    1) Do not insert new information, change the meaning of existing information, or remove significant pieces of information
    from the input text.
    2) You can replace complex medical terms with plain language equivalents or keep the medical term followed by
    a plain language definition.
    3) Your output must match the defined schema.
    """

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=ReadabilityEvaluation,
        temperature=0.2, # tweak temperature as needed
    )

    return client.chats.create(
        model="gemini-3.5-flash-lite",
        config=config
    )

def run_readability_agent(chat, message: str) -> ReadabilityEvaluation:
    """Sends a message to chat session and validates structured output."""

    # generate response
    response = chat.send_message(message)

    # parse pydantic model as JSON
    result = ReadabilityEvaluation.model_validate_json(response.text)


    # TODO: move this out of agent function?
    # verify textstat metrics w/ Python library
    computed_flesch_kincaid = textstat.flesch_kincaid_grade(result.revised_text)
    computed_smog = textstat.smog_index(result.revised_text)
    
    # print output
    print("Readability Scores")
    # print(f"Target Grade: {target_grade}")
    print(f"Computed Flesch-Kincaid grade level: {computed_flesch_kincaid:.2f}")
    print(f"Computed SMOG index: {computed_smog:.2f}")
    
    return result

In [17]:
# independent agent testing on sample text (generated by Gemini)
# TODO: revise this to make message sending smoother or add streaming
# note: can also use chat session for multiple turns if needed

simplified_text = """
The patient presented for routine evaluation. Vital signs reveal systolic blood pressure 
of 126 mmHg and diastolic blood pressure of 69 mmHg. Complete blood count demonstrated 
leukocyte count of 8.9 x 10^3/uL, hemoglobin concentration of 15.4 g/dL, and hematocrit 
of 46.5%. Prophylactic immunization against Hepatitis B was administered intramuscularly. 
Medication reconciliation was completed with no adverse contraindications noted.
"""
message = f"Rewrite the following clinical text to make it readable at or below the target grade level: \n\n{simplified_text}"

chat_session = create_readability_chat(target_grade=8)
output = run_readability_agent(chat_session, message)

print("Simplified Output: ")
print(output.revised_text)

print("\nBefore Revision: ")
print(f"Flesch-Kincaid grade level: {output.pre_flesch_kincaid_grade:.2f}")
print(f"SMOG index: {output.pre_smog:.2f}")

print("\nAfter Revision: ")
print(f"Flesch-Kincaid grade level: {output.post_flesch_kincaid_grade:.2f}")
print(f"SMOG index: {output.post_smog:.2f}")

Readability Scores
Computed Flesch-Kincaid grade level: 4.53
Computed SMOG index: 7.96
Simplified Output: 
The patient came in for a regular check-up. Blood pressure was 126 over 69. A complete blood count showed a white blood cell count of 8.9, hemoglobin of 15.4, and hematocrit of 46.5 percent. A shot to prevent Hepatitis B was given in the muscle. A review of all medications was done, and no harmful issues were found.

Before Revision: 
Flesch-Kincaid grade level: 12.40
SMOG index: 11.80

After Revision: 
Flesch-Kincaid grade level: 7.10
SMOG index: 7.50


**Important note**: Investigate discrepancy between computed Flesch-Kincaid grade level and post-revision score given by LLM.